In [1]:
# ===============================================
# Code Block 1 — Imports, Paths, and File Validation
# ===============================================
import os
import numpy as np
import pandas as pd

IN_DIR = "../data/processed"
EVENTS_FILE = os.path.join(IN_DIR, "events_cleaned.csv")
OUT_FILE = os.path.join(IN_DIR, "resource_features.csv")

# Basic checks
if not os.path.exists(EVENTS_FILE):
    raise FileNotFoundError(f"Missing cleaned events file: {EVENTS_FILE}")

print("Input file found:", EVENTS_FILE)

# Load events (timestamp must be datetime)
events = pd.read_csv(EVENTS_FILE, parse_dates=["timestamp"], low_memory=False)
print("Loaded events:", events.shape)

# Minimal validation
required = ["case_id", "timestamp", "activity"]
missing = [c for c in required if c not in events.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Ensure resource column exists
if "resource" not in events.columns:
    events["resource"] = "UNKNOWN"
else:
    # fill NA and normalize whitespace
    events["resource"] = events["resource"].fillna("UNKNOWN").astype(str).str.strip()

# Ensure events sorted by case + timestamp (important for shifts)
events = events.sort_values(["case_id", "timestamp"]).reset_index(drop=True)
print("Events sorted. Sample:")
display(events.head(8))


Input file found: ../data/processed\events_cleaned.csv
Loaded events: (262628, 14)
Events sorted. Sample:


,case_id,timestamp,activity,resource,cost,municipality,case_status,case_procedure,event_id,all_properties,year,month,weekday,hour
0,10002463,2014-08-05 00:00:00+00:00,register submission date request,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":136730,""labels"":[""Event""],""properties"":{...",2014,8,1,0
1,10002463,2014-08-06 00:00:00+00:00,regular procedure without MER,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137411,""labels"":[""Event""],""properties"":{...",2014,8,2,0
2,10002463,2014-08-06 00:00:00+00:00,create procedure confirmation,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137412,""labels"":[""Event""],""properties"":{...",2014,8,2,0
3,10002463,2014-08-06 00:00:00+00:00,publish,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137413,""labels"":[""Event""],""properties"":{...",2014,8,2,0
4,10002463,2014-08-06 00:00:00+00:00,create subcases completeness,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137414,""labels"":[""Event""],""properties"":{...",2014,8,2,0
5,10002463,2014-08-06 00:00:00+00:00,procedure change,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137415,""labels"":[""Event""],""properties"":{...",2014,8,2,0
6,10002463,2014-08-06 00:00:00+00:00,OLO messaging active,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137416,""labels"":[""Event""],""properties"":{...",2014,8,2,0
7,10002463,2014-08-06 00:00:00+00:00,send confirmation receipt,1550894,398.92447,BPIC15_4,O,Unknown,NaN,"{""id"":137417,""labels"":[""Event""],""properties"":{...",2014,8,2,0


In [2]:
# ===============================================
# Code Block 2 — Precompute per-event helper columns
# ===============================================
# Compute previous resource in same case (for handovers)
events["prev_resource"] = events.groupby("case_id")["resource"].shift(1)

# Flag whether actor changed (handover occurs between prev and current)
events["actor_changed"] = (events["resource"] != events["prev_resource"]) & events["prev_resource"].notna()

# Time gap from previous event in same case (seconds)
events["prev_gap_seconds"] = (
    events["timestamp"] - events.groupby("case_id")["timestamp"].shift(1)
).dt.total_seconds()

# Next event gap in same case (seconds) — helpful if needed
events["next_gap_seconds"] = (
    events.groupby("case_id")["timestamp"].shift(-1) - events["timestamp"]
).dt.total_seconds()

print("Computed helper columns: prev_resource, actor_changed, prev_gap_seconds, next_gap_seconds")
display(events[["case_id", "timestamp", "resource", "prev_resource", "actor_changed", "prev_gap_seconds"]].head(12))


Computed helper columns: prev_resource, actor_changed, prev_gap_seconds, next_gap_seconds


,case_id,timestamp,resource,prev_resource,actor_changed,prev_gap_seconds
0,10002463,2014-08-05 00:00:00+00:00,1550894,NaN,False,NaN
1,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,86400.0
2,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
3,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
4,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
5,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
6,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
7,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
8,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0
9,10002463,2014-08-06 00:00:00+00:00,1550894,1550894,False,0.0


In [3]:
# ===============================================
# Code Block 3 — Efficient handover counts (vectorized)
# ===============================================
# handover_in: event where actor_changed is True and current resource is r
# handover_out: event where actor_changed is True and previous resource is r

# Compute per-event boolean series
handover_in_events = events[events["actor_changed"] & events["resource"].notna()]
handover_out_events = events[events["actor_changed"] & events["prev_resource"].notna()]

# Count per resource
handover_in_counts = handover_in_events.groupby("resource").size().rename("handover_in")
handover_out_counts = handover_out_events.groupby("prev_resource").size().rename("handover_out")

print("Sample handover_in counts:")
display(handover_in_counts.head())
print("Sample handover_out counts:")
display(handover_out_counts.head())


Sample handover_in counts:


resource
10716070      7
1550894     900
2013365     793
2670601      61
2894257       2
Name: handover_in, dtype: int64

Sample handover_out counts:


prev_resource
10716070      7
1550894     900
2013365     793
2670601      61
2894257       2
Name: handover_out, dtype: int64

In [4]:
# ===============================================
# Code Block 4 — avg wait before event per resource
# ===============================================
# For each event, prev_gap_seconds is the wait before this event.
# For each resource, take mean of prev_gap_seconds where resource==r and prev_gap_seconds is finite.

wait_by_resource = (
    events.loc[events["prev_gap_seconds"].notna(), ["resource", "prev_gap_seconds"]]
    .groupby("resource")["prev_gap_seconds"]
    .mean()
    .rename("avg_wait_before_event_seconds")
)

print("Sample avg wait before event (seconds) per resource:")
display(wait_by_resource.head())


Sample avg wait before event (seconds) per resource:


resource
10716070    425633.684211
1550894     124427.269143
1946514      80938.909091
2013365      89306.013834
2670601     161186.050038
Name: avg_wait_before_event_seconds, dtype: float64

In [5]:
# ===============================================
# Code Block 5 — avg inter-event seconds per resource (within same case)
# ===============================================
# We compute diffs between consecutive events **of the same resource within the same case**.
# Method: group by (resource, case_id), sort timestamps and compute diffs; then average per resource.

def mean_inter_event_seconds(group):
    # group: DataFrame for a single (resource, case_id) with timestamps already sorted
    if len(group) <= 1:
        return np.nan
    diffs = group["timestamp"].diff().dt.total_seconds().iloc[1:]
    return diffs.mean()

# apply groupby on (resource, case_id) — returns many small groups; this is still efficient in pandas
res_case_means = (
    events.groupby(["resource", "case_id"])["timestamp"]
    .apply(lambda s: s.diff().dt.total_seconds().iloc[1:].mean() if len(s) > 1 else np.nan)
    .reset_index(name="inter_event_seconds")
)

# Now compute mean over cases for each resource
avg_inter_event = (
    res_case_means.groupby("resource")["inter_event_seconds"]
    .mean()
    .rename("avg_inter_event_seconds")
)

print("Sample avg inter-event seconds per resource:")
display(avg_inter_event.head())


Sample avg inter-event seconds per resource:


resource
10716070    550325.274725
1550894     188601.153028
1946514      77910.079309
2013365     128354.362852
2670601     174905.738252
Name: avg_inter_event_seconds, dtype: float64

In [6]:
# ===============================================
# Code Block 6 — Basic workload & activity diversity per resource
# ===============================================
n_events = events.groupby("resource").size().rename("n_events")
n_unique_cases = events.groupby("resource")["case_id"].nunique().rename("n_unique_cases")
n_distinct_activities = events.groupby("resource")["activity"].nunique().rename("n_distinct_activities")

# Compose a resource-level DataFrame
resource_features = (
    pd.concat([n_events, n_unique_cases, n_distinct_activities,
               handover_in_counts, handover_out_counts,
               wait_by_resource, avg_inter_event], axis=1)
    .fillna(0)  # fill missing numeric stats with 0 for resources with no counts
    .reset_index()
)

# Rename columns to ensure expected names exist
resource_features = resource_features.rename(columns={
    "index": "resource"
})

print("Resource features initial table shape:", resource_features.shape)
display(resource_features.head(10))


Resource features initial table shape: (64, 8)


,resource,n_events,n_unique_cases,n_distinct_activities,handover_in,handover_out,avg_wait_before_event_seconds,avg_inter_event_seconds
0,10716070,96,2,54,7.0,7.0,425633.684211,550325.274725
1,1550894,12875,169,217,900.0,900.0,124427.269143,188601.153028
2,1946514,180,4,63,0.0,0.0,80938.909091,77910.079309
3,2013365,12905,184,210,793.0,793.0,89306.013834,128354.362852
4,2670601,1363,45,146,61.0,61.0,161186.050038,174905.738252
5,2894257,9,1,8,2.0,2.0,889.111111,5966.500000
6,3069865,899,6,63,117.0,117.0,11853.390380,61604.625795
7,3069866,2933,20,83,285.0,285.0,11278.985930,41313.455941
8,3122446,284,3,70,43.0,43.0,46688.663121,152645.801598
9,3148844,105,1,60,18.0,18.0,1110.695238,80996.432692


In [7]:
# ===============================================
# Code Block 7 — Convert seconds → hours and add derived metrics
# ===============================================
for sec_col in ["avg_wait_before_event_seconds", "avg_inter_event_seconds"]:
    hr_col = sec_col.replace("_seconds", "_hours")
    resource_features[hr_col] = resource_features[sec_col] / 3600.0

# Add a per-event-per-case load metric: avg_events_per_case
resource_features["avg_events_per_case"] = (
    resource_features["n_events"] / resource_features["n_unique_cases"].replace({0: np.nan})
)

print("Converted to hours and added derived metrics. Sample:")
display(resource_features.head())


Converted to hours and added derived metrics. Sample:


,resource,n_events,n_unique_cases,n_distinct_activities,handover_in,handover_out,avg_wait_before_event_seconds,avg_inter_event_seconds,avg_wait_before_event_hours,avg_inter_event_hours,avg_events_per_case
0,10716070,96,2,54,7.0,7.0,425633.684211,550325.274725,118.231579,152.868132,48.000000
1,1550894,12875,169,217,900.0,900.0,124427.269143,188601.153028,34.563130,52.389209,76.183432
2,1946514,180,4,63,0.0,0.0,80938.909091,77910.079309,22.483030,21.641689,45.000000
3,2013365,12905,184,210,793.0,793.0,89306.013834,128354.362852,24.807226,35.653990,70.135870
4,2670601,1363,45,146,61.0,61.0,161186.050038,174905.738252,44.773903,48.584927,30.288889


In [8]:
# ===============================================
# Code Block 8 — Map resource → municipality safely (mode) and validate
# ===============================================
# A resource might appear in multiple municipalities; map to the most common one (mode)
res_to_muni = (
    events[events["municipality"].notna()]
    .groupby("resource")["municipality"]
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
    .rename("municipality")
)

# Merge the mapping into resource_features
resource_features = resource_features.merge(res_to_muni.reset_index(), on="resource", how="left")

# If municipality still missing, fill with "UNKNOWN_MUNI"
resource_features["municipality"] = resource_features["municipality"].fillna("UNKNOWN_MUNI")

print("Resource→municipality mapping sample:")
display(resource_features[["resource", "municipality"]].head(10))


Resource→municipality mapping sample:


,resource,municipality
0,10716070,BPIC15_1
1,1550894,BPIC15_4
2,1946514,BPIC15_3
3,2013365,BPIC15_3
4,2670601,BPIC15_1
5,2894257,BPIC15_3
6,3069865,BPIC15_3
7,3069866,BPIC15_3
8,3122446,BPIC15_3
9,3148844,BPIC15_3


In [9]:
# ===============================================
# Code Block 9 — Sanity checks and basic distributions
# ===============================================
print("Total resources:", resource_features.shape[0])
print("\nEvents per resource (describe):")
display(resource_features["n_events"].describe())

print("\nAvg wait before event (hours) - sample stats:")
display(resource_features["avg_wait_before_event_hours"].describe())

print("\nAvg inter-event (hours) - sample stats:")
display(resource_features["avg_inter_event_hours"].describe())


Total resources: 64

Events per resource (describe):


count       64.000000
mean      4103.562500
std       6432.617238
min          7.000000
25%         62.500000
50%        612.500000
75%       5897.500000
max      23700.000000
Name: n_events, dtype: float64


Avg wait before event (hours) - sample stats:


count     64.000000
mean      36.152104
std       35.000402
min        0.048713
25%        9.960297
50%       25.789265
75%       45.089285
max      139.612744
Name: avg_wait_before_event_hours, dtype: float64


Avg inter-event (hours) - sample stats:


count     64.000000
mean      79.418065
std       96.627654
min        1.657361
25%       34.067162
50%       59.384161
75%       91.459904
max      697.522361
Name: avg_inter_event_hours, dtype: float64

In [10]:
# ===============================================
# Code Block 10 — Save resource_features.csv
# ===============================================
resource_features.to_csv(OUT_FILE, index=False)
print(f"Saved resource features to: {OUT_FILE}")
print("Final shape:", resource_features.shape)


Saved resource features to: ../data/processed\resource_features.csv
Final shape: (64, 12)


In [ ]:
# Replace case_df with your case-level DataFrame name
# Ensure each column below exists in that DataFrame

case_metric_cols = [
    "case_cycle_time",          # case cycle time
    "active_processing_time",   # active processing time
    "waiting_time",             # waiting / idle time
    "structural_complexity",    # structural complexity per case
    "total_case_cost",          # total cost per case
    "event_count",              # number of events per case
    "case_hybrid_score",        # hybrid performance score per case
    "case_outlier_score",       # or case_efficiency_index, etc.
]

case_metric_titles = [
    "Case Cycle Time",
    "Active Processing Time",
    "Waiting Time",
    "Structural Complexity",
    "Total Cost per Case",
    "Event Count per Case",
    "Case Hybrid Performance Score",
    "Case Outlier / Efficiency Score",
]

fig_case, axes_case = plot_metric_dashboard(
    df=case_df,
    metric_cols=case_metric_cols,
    titles=case_metric_titles,
    n_cols=4,  # 2x4 grid → 8 tiles
    suptitle="Case Performance Dashboard",
    x_axis_label="Case index (sorted by metric)",
)